# Metody wstępnego przetwarzania danych, semestr 2026L

## Lab 10. Metody wstępnego przetwarzania tekstu na potrzeby Natural Language Processing (NLP), część 1

### 1. Etapy przetwarzania tekstu

Podobnie jak w przypadku obrazów, surowy tekst pochodzący z internetu, książek czy dokumentów jest pełen informacyjnego "szumu". Algorytmy uczenia maszynowego nie potrafią samodzielnie czytać liter – rozumieją wyłącznie liczby. Dlatego tekst przed analizą musi przejść przez rygorystyczny proces czyszczenia i transformacji.

Oto standardowe etapy przetwarzania języka naturalnego (NLP):

#### 1. Czyszczenie tekstu (Text Cleaning)

To pierwszy i najbardziej podstawowy etap, mający na celu usunięcie zbędnych elementów, które mogłyby zaburzyć analizę.

* **Lowercasing:** Zamiana wszystkich liter na małe. Dzięki temu maszyna nie potraktuje słów "Kot" i "kot" jako dwóch zupełnie różnych pojęć.
* **Usuwanie znaków specjalnych i interpunkcji:** Wycinanie znaków typu `@`, `#`, `!`, chyba że niosą one kluczową informację (np. hashtagi w analizie postów z mediów społecznościowych).
* **Usuwanie tagów HTML i URL:** Niezbędne przy oczyszczaniu danych pobranych bezpośrednio ze stron internetowych (tzw. web scraping).

#### 2. Tokenizacja (Tokenization)

Podział długiego ciągu znaków (np. całego dokumentu, akapitu lub zdania) na mniejsze, operacyjne jednostki, nazywane **tokenami**.

* **Tokenizacja słów:** Klasyczne dzielenie tekstu po spacjach i znakach interpunkcyjnych (np. zdanie "Ala ma kota" zostaje rozbite na listę: `["Ala", "ma", "kota"]`).
* **Tokenizacja subwordów (pod-słów):** Metoda wykorzystywana przez nowoczesne modele (jak GPT czy BERT). Dzieli rzadsze słowa na mniejsze fragmenty (np. rdzenie i przedrostki), co świetnie pomaga radzić sobie z literówkami, odmianami wyrazów i zupełnie nowymi słowami.

#### 3. Usuwanie tzw. "stop words"

W każdym języku istnieją słowa, które pełnią funkcję wyłącznie gramatyczną i występują bardzo często, ale nie niosą ze sobą konkretnego znaczenia w kontekście tematyki tekstu (w języku polskim to np. "w", "na", "i", "oraz", "że"). Ich usunięcie zmniejsza rozmiar danych i pozwala modelowi skupić się wyłącznie na słowach kluczowych dla sensu wypowiedzi.

#### 4. Sprowadzanie do formy podstawowej (Stemming i Lematyzacja)

Ponieważ słowa mogą występować w wielu formach gramatycznych (co jest szczególnie widoczne w języku polskim, który ma bogatą fleksję), sprowadza się je do wspólnego mianownika, aby model wiedział, że mówimy o tym samym koncepcie.

* **Stemming:** Szybkie, algorytmiczne "obcinanie" końcówek wyrazów, aby zostawić sam rdzeń (np. "bieganie", "biegać", "biegacz" stają się rdzeniem "bieg"). Bywa brutalny i czasem tworzy słowa, które w ogóle nie istnieją.
* **Lematyzacja:** Znacznie inteligentniejsze podejście. Korzysta z reguł i słowników językowych, aby sprowadzić słowo do jego poprawnej formy słownikowej (tzw. lematu). Na przykład słowa "byłem" i "będę" zostaną zamienione na "być".

#### 5. Znakowanie części mowy (POS Tagging)

To etap, w którym algorytm przypisuje każdemu słowu odpowiednią część mowy (rzeczownik, czasownik, przymiotnik itp.). Jest to niezwykle przydatne przy rozwiązywaniu wieloznaczności – na przykład słowo "zamek" może oznaczać budowlę lub zamek w drzwiach. Rozpoznanie, w jakim otoczeniu gramatycznym występuje to słowo, pomaga modelowi zinterpretować jego właściwy sens.

#### 6. Wektoryzacja (Vectorization / Word Embeddings)

To ostateczny i najważniejszy krok przejścia ze świata lingwistyki do świata matematyki. Ponieważ sieci neuronowe przyjmują na wejściu wyłącznie wektory liczbowe, każdy przygotowany wcześniej token musi zostać zakodowany do postaci liczb.

* **Klasyczne metody (np. Bag of Words, TF-IDF):** Zliczają, jak często dane słowo występuje w dokumencie w stosunku do całego zbioru tekstów.
* **Nowoczesne metody (Word Embeddings - np. Word2Vec, architektury Transformer):** Tworzą gęste, wielowymiarowe wektory, w których słowa o podobnym znaczeniu znajdują się blisko siebie w przestrzeni matematycznej. Dzięki temu model "wie", że słowo "król" ma się do "mężczyzny" tak, jak "królowa" do "kobiety".

### 2. Wyrażenie ragularne jako ważny komponent przetwarzania tekstu

Dane w postaci tekstu często nazywamy danymi nieustrukturyzowanymi jeżeli mówimy o danych pochodzących z książek, stron internetowych, dokumentów. Wciąż możemy tam odnaleźć słowa, zdania, ale często znajdują się one w otoczeniu dodatkowych elementów takich jak znaczniki np. html, znajdziemy dużo kolokwializmów, literówek itp. Im bardziej precyzyjnie oczyścimy (ekstrakcja) taki tekst, tym lepsze efekty możemy uzyskać w naszym modelu.

Zadania ekstrakcji danych składają się bardzo często z wielu skomplikowanych reguł. Można by je umieścić w ogromnym drzewie decyzyjnym reguł `IF ... ELIF ... ELSE`, ale po pierwsze byłoby to trudne i na pewno bardzo skomplikowane, również przy aktualizacji takiego zbioru reguł.

Do takich zadań najczęściej wykorzystywane są wyrażenia regularne (ang. regular expression, często w skrócie regexp), które pozwalają zdefiniować reguły, które "łapią" wiele wzorców na raz dzięki m.in. kwantyfikatorom.

W tym laboratoium zaprezentowane zostaną wybrane elementy wyrażeń regularnych i ich wykorzystanie w języku Python.

## 3. Wyrażenia regularne w Pythonie i moduł `re`

Wyrażenia regularne zapisujemy w formie znaków, które niosą ze sobą spacjalne znaczenie np. szukanej frazy, ale również dopuszczalnych znaków czy ich powieleń.

Na początek wybrane metaznaki wyrażeń regularnych.

**Tabela 1**

| Metaznak | Opis | Przykład | 
| --- | --- | --- |
| `$`| Kończy się na | `"ski$"` - nazwa kończy się znakami `ski` |
| `^`| Rozpoczyna się od | `"^Pol"` - nazwa rozpoczyna się znakami `Pol` |
| `.`| Dowolny znak (oprócz znaku nowej linii) | `"b.r"` - nazwa rozpoczyna się od litery `b` następnie zwiera dowolny znak i kończy się literą `r` |
| `\`| Znak ucieczki lub rozpoczęcie specjalnej sekwencji | `"^\."` - nazwa rozpoczyna się kropką (która również jest metaznakiem) |
| `*` | Zero lub więcej wystąpień | `"b.*o"` - pasują `bo`, `bro`, `booo` itd. |
| `+` | Co najmniej jedno wystąpienie | `"b.+o"` - pasują `bro`, `boo` itd., nie pasuje `bo` |
| `?` | Co najwyżej jedno wystąpienie | `"b.?o"` - pasują `bo`, `bro`, `boo`, ale nie pasuje `booo` |
| `[]` | Określa zbiór dopuszczalnych wartości | `"[a-z]"` - wszystkie znaki między `a` oraz `z`, `"[0-9]"` - cyfry od 0 do 9, `"[a-c0-3]"` - znaki od `a` do `c` oraz cyfry od 0 do 3. |
| `[^]` | Określa zbiór wartości, które nie mogą znaleźć się w wynikach | `"[^abc]"` - wartości, które nie zawierają `a`, `b` lub `c`, `"[^0-9]"` - wartości, które nie zawierają cyfr |
| `()` | Pozwala na grupowanie wyrażeń. | |
| `\|` | Alternatywa | `"tak\|Tak"` - słowo `tak` lub `Tak`, można również zapisać jako `"(T\|t)ak"` |
| `{m,n}` | Określa bardziej precyzyjnie liczebność wystąpienia elementów wyrażenia | `"b.{2}o"` - dokładnie dwa dwolne znaki, `"b.{2,3}"` - co najmniej 2, ale nie więcej niż 3 wystąpienia, `"b.{2,}o"` - co najmniej dwa wystąpienia, `"b.{,3}"` - co najwyżej 3 wystąpienia. |

Standardowa biblioteka Pythona zawiera moduł `re`, który dostarcza kilka funkcji pozwalających na obsługę wyrażeń regularnych. 

**Dokumentację** zawierającą elementy składniowe wyrażeń regularnych oraz funkcje tego modułu można znaleźć tu: https://docs.python.org/3/library/re.html

In [2]:
# import modułu
import re

### 3.1 Funkcja `re.match`

Sygnatura: `re.match(pattern, string, flags=0)`

Zwraca obiekt `Match` jeżeli na początku łańcucha znaków (`string`) zawiera zero lub więcej pasujących elementów do wyrażenia (`pattern`). Nie znajduje wszystkich wystąpień wyrażenia (to można osiągnąć poprzez użycie `re.search()`)

**Przyład 1**

In [3]:
# rozpoczyna się od dowolnego znaku, a drugi znak to o
result = re.match(r'^.o', 'Kowalski')
type(result)

re.Match

In [4]:
# obiekt Match zawiera informacje o tym jaka część łańcucha pasuje do wzorca
print(result)
print(result[0])

<re.Match object; span=(0, 2), match='Ko'>
Ko


In [ ]:
# gdybyśmy chcieli sprawdzić czy nazwa kończy się na 'ski` to
# wykorzystując poniższe wyrażenie - nie zadziała jak byśmy tego chcieli
# match dopasowuje tylko na początku wyrażenia
result = re.match(r'ski$', 'Kowalski')
result

In [5]:
# ale kombinując nieco inaczej, możemy osiągnąć tutaj porządany efekt
result = re.match('.+ski', 'Kowalski')
result

<re.Match object; span=(0, 8), match='Kowalski'>

In [6]:
result = re.match(r'.+ski', 'Jan Kowalski')
result

<re.Match object; span=(0, 12), match='Jan Kowalski'>

In [7]:
# w dokumentacji funkcji match wyczytamy również, że dopasowanie zostanie wykonane
# tylko dla pierwszej linii
result = re.match(r'.+ski', 'Jan Kowalski \n Adam Malinowski')
result

<re.Match object; span=(0, 12), match='Jan Kowalski'>

### 3.2 Funkcja `re.fullmatch`

**Przykład 2**

In [43]:
# podobnie jak match, ale dopasowuje wzorzec do całej sekwencji, a nie szuka tylko pasującego fragmentu
# a to dlatego, że mamy zarówno metaznacznik rozpoczyna się od (to po ^) oraz kończy się na (to przed $) 
result = re.fullmatch(r'^[A-Z].+ski$', 'Kowalski')
result

<re.Match object; span=(0, 8), match='Kowalski'>

In [13]:
result = re.fullmatch(r'^[A-Z].+ski$', 'Kowalski Jan')
result

### 3.3 Funkcja `re.compile`

Funkcja re.compile pozwala na kompilację wyrażenia do obiektu wyrażenia regularnego (więcej [tu](https://docs.python.org/3/library/re.html#re-objects)), an którym możemy później wywołać metody takie jak `match()` czy `search()` oraz inne. Możemy również zdefiniować dodatkowe flagi dla tego wyrażenia.

In [14]:
pattern = r'^.o'

regexp = re.compile(pattern)
result = regexp.match('Kowalski')

# to to samo co wcześniejszy przykład
# result = re.match('^.o', 'Kowalski')

# jednak jeżeli chcemy to wyrażenie wykonać wielokrotnie w naszym skrypcie, to po kompilacji może to być bardziej efektywne
result

<re.Match object; span=(0, 2), match='Ko'>

### 3.4 Funkcja `Pattern.search`

Sygnatura: `Pattern.search(string[, pos[, endpos]])`

Ta funkcja odnajduje pierwsze dopasowanie wzorca w zadanym łańcuchu znaków. Może przyjmować opcjonalne argumenty określające zakres w sekwencji do przeszukania (`sekwencja[pos:endpos]`).


In [15]:
result = regexp.search('Kowalski')
result

<re.Match object; span=(0, 2), match='Ko'>

In [16]:
pattern = r'[A-Z]'
regexp = re.compile(pattern)
result = regexp.search('Kowalski')
result

<re.Match object; span=(0, 1), match='K'>

In [18]:
result = regexp.search('Kowalski', 1)
result

In [19]:
result = regexp.search('KOalski', 1)
result

<re.Match object; span=(1, 2), match='O'>

### 3.5 Funkcja `re.split`

Podobnie jak funkcja `str.split()` dzieli sekwencje łańcucha znaków z wykorzystaniem podanego podłańcucha, tak `re.split()` dzieli łańuch z wykorzystaniem wyrażenia regularnego. Poprzez zdefiniowanie wartości argumentu `maxsplit` możemy również ograniczyc liczbę podziałów, po których funkcja powinna przestać to robić. Zwraca listę.

In [22]:
seq = '1 Abracadabra 2 to 3 czary 4 i 5 magia'
result = re.split(r'[0-9]', seq)
result

['', ' Abracadabra ', ' to ', ' czary ', ' i ', ' magia']

In [23]:
result = re.split(r'[0-9]', seq, maxsplit=3)
result

['', ' Abracadabra ', ' to ', ' czary 4 i 5 magia']

### 3.6 Funkcja `re.findall`

Zwraca wszystkie wystąpienia wzorca w łańcuchu znaków jako listę łańcuchów lub krotek.

In [27]:
re.findall(r'k[a-z]*', 'Ala ma kota, a kot to Filemon.')

['kota', 'kot']

In [28]:
re.findall(r'[0-9]', '1 Abracadabra 2 to 3 czary 4 i 5 magia')

['1', '2', '3', '4', '5']

In [32]:
# jedną z flag, którą możemy przekazać w celu zmiany domyślnego sposobu ewaluacji wyrażenia
# jest re.MULTILINE, które jest przydatne dla przeszukiwania tekstu wielowierszowego (oddzielonego znakami nowego wiersza)

# bez flagi MULTILINE
res = re.findall(r'^[A-Z].*', 'Bob był budowniczym\nNie bo nie\nco tam słychać?\n')
print(res)

# z flagą MULTILINE
res = re.findall(r'^[A-Z].+', 'Bob był budowniczym\nNie bo nie\nco tam słychać?\n', flags=re.MULTILINE)
print(res)

['Bob był budowniczym']
['Bob był budowniczym', 'Nie bo nie']


In [33]:
# mamy też inne flagi, np. IGNORECASE - wielkość znaków nie jest brana pod uwagę

# z flagą
res = re.findall(r'bob', 'Bob był budowniczym\nNie bo nie\nco tam słychać?\n', flags=re.IGNORECASE)
print(res)

['Bob']


### 3.7 Funkcja `re.finditer`

Funkcja ta zwraca iterator, który zwraca obiekty Match dla każdego odnalezionego dopasowania w łańcuchu znaków.

In [30]:
result = re.finditer('[0-9]', '1 Abracadabra 2 to 3 czary 4 i 5 magia')
result

In [31]:
for res in result:
    print(res)

<re.Match object; span=(0, 1), match='1'>
<re.Match object; span=(14, 15), match='2'>
<re.Match object; span=(19, 20), match='3'>
<re.Match object; span=(27, 28), match='4'>
<re.Match object; span=(31, 32), match='5'>


### 4. Kilka przykładów wyrażeń regularnych

In [50]:
# szukamy 3-ech kolejno występujących po sobie cyfr
tekst = '1. Agent 007 James Bond z pokoju numer 04'
# tak jest bardzo uciążliwie
print(re.findall(r'[0-9][0-9][0-9]', tekst))

# po to mamy kwalifikatory
print(re.findall(r'[0-9]{3}', tekst))

['007']
['007']


In [52]:
# liczby jedno i dwucyfrowe
tekst = '1. Agent 007 James Bond z pokoju numer 04'
re.findall(r'[0-9]{1,2}', tekst, flags=re.MULTILINE)

['1', '00', '7', '04']

In [53]:
# liczby jedno i dwucyfrowe (z metaznakiem \w - do wyjaśnienia)
tekst = '1. Agent 007 James Bond z pokoju numer 04'
re.findall(r'\w[0-9]{1,2}', tekst, flags=re.MULTILINE)

['007', '04']

In [54]:
# liczby jedno i dwucyfrowe (z metaznakiem \w - do wyjaśnienia)
# oraz grupowaniem, aby to \w zostało zaaplikowane do całego wyrażenia
tekst = '1. Agent 007 James Bond z pokoju numer 04'
re.findall(r'\w([0-9]{1,2})', tekst, flags=re.MULTILINE)

['07', '4']

### 5. Zadania

In [38]:
# tekst do wykorzystania w zadaniach
text = """
Adam Malinowski
.gitignore
2023-01-17 error "Page not found"
[2025-03-06] NOTICE "User admin logged in"
Code 3300 was invalid
https://www.onet.pl 200 176353
File /etc/passwd: permission denied
Józef
Ania
JOLA
marek
Kowalski
bodo363
PIN 0000 was invalid
/users/test is not a valid directory name
192.168.0.1 access denied
1000
666
"""

Zadania wykonaj na zmiennej `text` zadeklarowanej powyżej.


1. Wypisz wszystkie dopasowania dla linii rozpoczynających się wielką literą.
2. Wypisz dopasowania zawierające cyfry.
3. Wypisz całe linie zawierające kropkę.
4. Wypisz liczby składające się z co najmniej 3 cyfr.
5. Wypisz całe linie zawierające liczby składające się z co najmniej 3 cyfr.
6. Wypisz całe linie zawierające tylko litery.
7. Wypisz całe linie zawierające tylko cyfry.
8. Wypisz dopasowania zawierające słowo `valid` lub `invalid`.
9. Wypisz dopasowania zawierające datę w formacie `YYYY-MM-DD`.
10. Wypisz dopasowania zawierające ścieżkę w formacie UNIX (/.../...).
11. Wypisz dopasowania zawierającą adres IP w wersji 4.
12. Wypisz tekst z każdego cytowania (tekst pomiędzy " oraz ").
13. Wypisz linie, których długość to dokładnie 4 znaki.